### Verify Parties Schema

This cell verifies that the DataFrame has the expected column names and Spark data types.

In [0]:
# Import commonly used libraries.

import pyspark.sql.functions as F
import datetime
import pandas as pd
import dateutil
import json

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    IntegerType,
    TimestampType,
    DateType,
    DecimalType,
    DoubleType,
    BooleanType
)

# this imports:
# PySpark functions
# datetime utilities
# json parser
# Spark SQL schema/data types

### Define Base Paths

These constants centralize ADLS paths.

Keeping paths here avoids hardcoding long ADLS paths in every raw notebook.

In [0]:
# Base ADLS paths used throughout the project.

ADLS_DEV_BASE_PATH = "abfss://oaonoperationsdev@stoncomdev001.dfs.core.windows.net/"
SOURCE_TABLES_PATH = "oaon-sandbox.operations.dynamics.com/Tables/"
DELTALAKE_RAW_PATH = "DeltaLake/Raw/"

# Example final source path:
# ADLS_DEV_BASE_PATH + SOURCE_TABLES_PATH + "Purchase/Parties/"

# Example final Delta raw path:
# ADLS_DEV_BASE_PATH + "DeltaLake/Raw/Purchase/Parties"

### Read Secrets And Configure ADLS OAuth

This cell reads service principal secrets and configures Spark OAuth access to the ADLS storage account.

This is included here because most project notebooks will run `SharedLibraries` before reading or writing ADLS data.

In [0]:
# Read service principal credentials from Databricks secret scope.

service_credential = dbutils.secrets.get(scope="kv-oncom-dev-scope",key="client-secret")
appid = dbutils.secrets.get(scope="kv-oncom-dev-scope",key="app-id")
tenantid = dbutils.secrets.get(scope="kv-oncom-dev-scope",key="tenant-id")

# Configure Spark OAuth access to ADLS Gen2.

spark.conf.set("fs.azure.account.auth.type.stoncomdev001.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.stoncomdev001.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.stoncomdev001.dfs.core.windows.net", appid)
spark.conf.set("fs.azure.account.oauth2.client.secret.stoncomdev001.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.stoncomdev001.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenantid}/oauth2/token")

# This makes sure any notebook that runs SharedLibraries can access ADLS.
# This is practical because every raw notebook needs to read/write ADLS.

### Map CDM Data Types To Spark Data Types

This function converts CDM `dataFormat` values from `.cdm.json` files into Spark SQL data types.

This is the key replacement for the CDM connector's automatic schema handling.

In [0]:
# Convert CDM dataFormat values to Spark SQL data types.

def getSparkType(dataFormat):
    if dataFormat == "Int64":
        return LongType()
    elif dataFormat == "Int32":
        return IntegerType()
    elif dataFormat == "String":
        return StringType()
    elif dataFormat == "DateTime":
        return TimestampType()
    elif dataFormat == "Date":
        return DateType()
    elif dataFormat == "Decimal":
        return DecimalType(28, 6)
    elif dataFormat == "Double":
        return DoubleType()
    elif dataFormat == "Boolean":
        return BooleanType()
    else:
        return StringType()
    

# This maps CDM metadata types to Spark types.

# Examples:
# Int64    -> LongType
# String   -> StringType
# DateTime -> TimestampType
# Decimal  -> DecimalType(28, 6)

# If an unknown type appears, it defaults to StringType.

### Build Entity Schema From CDM JSON

This function reads an entity `.cdm.json` file and converts `definitions[0].hasAttributes` into a Spark StructType schema.

This avoids writing StructType manually for every table.

In [0]:
# Build Spark StructType schema from entity .cdm.json file.

def getEntitySchema(manifestPath, entity):
    cdmPath = ADLS_DEV_BASE_PATH + SOURCE_TABLES_PATH + manifestPath + "/" + entity + ".cdm.json"

    cdmContent = dbutils.fs.head(cdmPath, 1000000)

    cdmJson = json.loads(cdmContent)

    attributes = cdmJson["definitions"][0]["hasAttributes"]

    fields = []

    for attr in attributes:
        columnName = attr["name"]
        dataFormat = attr.get("dataFormat", "String")
        sparkType = getSparkType(dataFormat)

        fields.append(StructField(columnName, sparkType, True))

    schema = StructType(fields)

    return schema


# For example:
# getEntitySchema("Purchase", "Parties")

# reads:
# Tables/Purchase/Parties.cdm.json

# Then extracts:
# definitions[0].hasAttributes

# Then builds a Spark schema.


### Read Entity

This function replaces the instructor's `spark.read.format("com.microsoft.cdm")` logic.

It reads the entity schema from `.cdm.json`, then reads the matching headerless CSV folder with that schema.

In [0]:

# Read CDM-style entity data without using the com.microsoft.cdm connector.

def readEntity(manifestPath, entity):
    entitySchema = getEntitySchema(manifestPath, entity)

    entityPath = ADLS_DEV_BASE_PATH + SOURCE_TABLES_PATH + manifestPath + "/" + entity + "/"

    df = (spark.read.format("csv")
    .schema(entitySchema)
    .option("header", "false")
    .option("delimiter", ",")
    .option("escape", '"')
    .option("path", entityPath)
    .load())

    return df

# this is the most important function.

# Example:
# partiesdf = readEntity("Purchase", "Parties")

# It reads:
# Tables/Purchase/Parties.cdm.json
# Tables/Purchase/Parties/

# and returns a DataFrame with correct column names and Spark types.

### Write Raw Entity Data To Delta Lake

This function writes a DataFrame to ADLS in Delta format.

The raw Delta layer is stored outside the original Dynamics source folder.

In [0]:
# Write entity DataFrame to raw Delta Lake path.

def writeRawToDeltaLake(entityDf, deltaLakePath):
    entityDf.write.mode("overwrite").option("overwriteSchema","True").option("path", ADLS_DEV_BASE_PATH + deltaLakePath).save()


# Example:
# writeRawToDeltaLake(partiesdf, "DeltaLake/Raw/Purchase/Parties")

# writes Delta files to:
# abfss://oaonoperationsdev@stoncomdev001.dfs.core.windows.net/DeltaLake/Raw/Purchase/Parties
# This creates a raw Delta version of the source data.

### Read Data From Raw Delta Path

This function reads Delta files from the raw Delta Lake path.

It is used after raw data has already been written to Delta.

In [0]:
# Read an entity from the raw Delta Lake path.

def readFromDeltaPath(entityName):
    df = (spark.read.format("delta")
      .option("path", f"{ADLS_DEV_BASE_PATH}{DELTALAKE_RAW_PATH}{entityName}")
      .load()
      )
    return df

# Example:

# readFromDeltaPath("Purchase/Parties")

# reads:

# DeltaLake/Raw/Purchase/Parties

### Save DataFrame To Catalog Table

This function saves a DataFrame as a managed Delta table.

Use this only when the cluster/catalog setup supports the target catalog or schema.

In [0]:
# Save DataFrame as a Delta table in a schema.

def saveDeltaTableToCatalog(df, schema, tableName):
    schema = schema.lower()
    tableName = tableName.lower()
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    df.write.format("delta").mode("overwrite").saveAsTable(f"{schema}.{tableName}")

# This function creates a schema if it does not exist, then saves the DataFrame as a table.

# For now, raw notebooks write to ADLS Delta paths first.